# 5. Data governance: licences, land, Indigenous data: solutions

Exercises of [chapter 5](../../05_data_governance.md). Run the setup cell first. The exercises change a *copy* of the sample's site registry, never the real one.

These are the solutions; try the exercise notebook first.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import copy, datetime, tempfile
import yaml
from data_acquisition.catalog import build, descriptors
from data_acquisition.catalog import summary as summaries
from data_acquisition.readers import eml
from data_acquisition.steps.curate import curate_item

REFERENCE = yaml.safe_load((ROOT / "docs/manual/solutions/training_pond.yaml").read_text())
MY_REGISTRY = Path(tempfile.mkdtemp()) / "sites.yaml"
MY_REGISTRY.write_text((SAMPLE / "sites.yaml").read_text())

**1. Licences from records.** Read the sample's `metadata.xml` with `eml.read` and print its licence. Change the licence URL in the text to CC BY-SA 4.0, then to a licence the reader does not know: what does it return?

> **From the chapter.** §1: each record format maps licences to SPDX identifiers, with its own table.
>
> **Code.** `eml.read` and its table `eml.LICENCES`, in [`data_acquisition/readers/eml.py`](../../../../data_acquisition/readers/eml.py).

In [ ]:
text = (DATASET / "metadata.xml").read_text()
print("as published:", eml.read(text)["licence"])
by_sa = text.replace("licenses/by/4.0", "licenses/by-sa/4.0")
print("CC BY-SA:    ", eml.read(by_sa)["licence"])
unknown = text.replace("https://creativecommons.org/licenses/by/4.0/", "https://example.org/my-own-terms")
print("unknown:     ", eml.read(unknown)["licence"], "| the text is kept:", eml.read(unknown)["rights"][:60])
# A licence the reader does not know is None: left empty and listed as missing, never guessed.

**2. A registry entry.** In `MY_REGISTRY` (a copy of the sample's registry), set `land_status` to `{value: ..., by: <you>, date: <today>, note: ...}` and check it with `summaries.check_sites`. Then try an entry without `by`.

> **From the chapter.** §4: a manual field in the registry is either `null` or `{value, by, date[, note]}`.
>
> **Code.** `summaries.check_sites`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py). `MY_REGISTRY` is defined above: a copy of the sample's registry.

In [ ]:
registry = yaml.safe_load(MY_REGISTRY.read_text())
site = registry["sites"]["training_pond"]
site["land_status"] = {"value": "private ranch (invented)", "by": "A Student", "date": datetime.date.today(),
                       "note": "from the landowner, by phone"}
print("valid entry:", summaries.check_sites(registry["sites"]) or "no problems")
bad = copy.deepcopy(registry["sites"]); bad["training_pond"]["land_status"] = {"value": "private"}
print("without by: ", summaries.check_sites(bad))
MY_REGISTRY.write_text(yaml.safe_dump(registry, sort_keys=False))

**3. From registry to items.** Build and curate the sample with your registry (point `BEAVER_SITES` to it). What does the orthomosaic item say about `land_status`, and where did the value come from (`beaver:origin`)? What changes if `by` is `claude`?

> **From the chapter.** §4: `curate` copies the registry into the items and records who entered each value. `beaver:origin` is described in [chapter 4](../../04_catalogs_and_metadata.md), §2.
>
> **Code.** `summaries.load` in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py), `build.build` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py), and `curate_item` in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py).

In [ ]:
def orthomosaic_item():
    s = summaries.load(str(ROOT / "docs/manual/solutions/training_pond.yaml"))
    catalog, _ = build.build(SAMPLE, s)
    item = next(i for i in catalog.get_items(recursive=True) if i.id.endswith("orthomosaic"))
    curate_item(item, s)
    return item

os.environ["BEAVER_SITES"] = str(MY_REGISTRY)
item = orthomosaic_item()
print(item.properties["beaver:land_status"], item.properties["beaver:origin"]["land_status"])
registry = yaml.safe_load(MY_REGISTRY.read_text())
registry["sites"]["training_pond"]["land_status"]["by"] = "claude"
MY_REGISTRY.write_text(yaml.safe_dump(registry, sort_keys=False))
print(orthomosaic_item().properties["beaver:origin"]["land_status"])
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")
# Entered by a person: source "manual", with who and when. Entered by Claude: source "claude", never confirmed.

**4. Hints only.** Add `land_status: private` under the site in a copy of the reference summary, next to `hints`. What does the check say?

> **From the chapter.** §4: summaries may only suggest values for a site.
>
> **Code.** `summaries.check` and `summaries.load_sites`, in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py). `REFERENCE` is defined above: the reference summary, as written.

In [ ]:
s = copy.deepcopy(REFERENCE)
s["sites"]["training_pond"]["land_status"] = "private"
print(summaries.check(s, summaries.load_sites()))
# Blocks: a summary may only hint; land status is decided in the site registry, by a person.

**5. Find it in the code.** Where does `curate` copy the registry's fields into items, and where does a site collection's licence become `various`?

> **From the chapter.** §4 for what is copied, §1 for the collection's licence.
>
> **Code.** Look in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py) and [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py).

In [ ]:
import inspect
from data_acquisition.steps import curate
for fn, needle in ((curate.curate_item, "_set_entered"), (build.build, '"various"')):
    source, first = inspect.getsourcelines(fn)
    for n, line in enumerate(source, first):
        if needle in line:
            print(f"{inspect.getsourcefile(fn)}:{n}: {line.strip()}")